In [10]:
import regex as re

#### Instead of using pre-made regex from opensource models, one should be able to create one from scratch and for that; one should know the internals. So let's start from the beginning

##### Remember the goal of a regex before tokenization is to chunk words in a meaningful manner so that BPE can only merge inside these chunks and learn meaningful merges instead of merging across word boundries.

## Let's start with a basic regex and we'll improve it one step at a time

In [36]:
samples = [
    "Hello.... world!!!!",
    "I'm learning BPE. 45320000",
    "GPT-4 costs 200,000,000.0 euros.",
    "Hallo, ich-lerne Deutsch.",
    "(Hello)   'world'\nHow are you?",
    "こんにちは世界",
]

In [37]:
def apply_regex(regex_p, samples):
    pattern = re.compile(regex_p)
    for sample in samples:
        print(f"original string: {sample}")
        chunks = pattern.findall(sample)
        print(f"chunks: {chunks}")



In [38]:
pattern = r'\p{L}+' # one or more unicode letters
apply_regex(pattern, samples)

original string: Hello.... world!!!!
chunks: ['Hello', 'world']
original string: I'm learning BPE. 45320000
chunks: ['I', 'm', 'learning', 'BPE']
original string: GPT-4 costs 200,000,000.0 euros.
chunks: ['GPT', 'costs', 'euros']
original string: Hallo, ich-lerne Deutsch.
chunks: ['Hallo', 'ich', 'lerne', 'Deutsch']
original string: (Hello)   'world'
How are you?
chunks: ['Hello', 'world', 'How', 'are', 'you']
original string: こんにちは世界
chunks: ['こんにちは世界']


#### At this point we are able to catch words. what about the numbers? punctuations? contractions like i'm? spaces, newline?

similar to the one which catches letters we also have a pattern to catch unicode numbers and spaces i.e '\p{N}+'. for whitespace char '\s+'. + means one or more here

In [39]:
pattern = r'\p{N}+'
apply_regex(pattern, samples)

original string: Hello.... world!!!!
chunks: []
original string: I'm learning BPE. 45320000
chunks: ['45320000']
original string: GPT-4 costs 200,000,000.0 euros.
chunks: ['4', '200', '000', '000', '0']
original string: Hallo, ich-lerne Deutsch.
chunks: []
original string: (Hello)   'world'
How are you?
chunks: []
original string: こんにちは世界
chunks: []


In [40]:
pattern = r'\s+' # white spaces and newline characters
apply_regex(pattern, samples)

original string: Hello.... world!!!!
chunks: [' ']
original string: I'm learning BPE. 45320000
chunks: [' ', ' ', ' ']
original string: GPT-4 costs 200,000,000.0 euros.
chunks: [' ', ' ', ' ']
original string: Hallo, ich-lerne Deutsch.
chunks: [' ', ' ']
original string: (Hello)   'world'
How are you?
chunks: ['   ', '\n', ' ', ' ']
original string: こんにちは世界
chunks: []


#### Great! now we can capture spaces, letters, numbers. what about Punctuations? We can combine and negate all of the above patterns to find any unicode characters that are not numbers, letters or white spaces

In [41]:
pattern = r"[^\s\p{L}\p{N}]+"
apply_regex(pattern, samples)

original string: Hello.... world!!!!
chunks: ['....', '!!!!']
original string: I'm learning BPE. 45320000
chunks: ["'", '.']
original string: GPT-4 costs 200,000,000.0 euros.
chunks: ['-', ',', ',', '.', '.']
original string: Hallo, ich-lerne Deutsch.
chunks: [',', '-', '.']
original string: (Hello)   'world'
How are you?
chunks: ['(', ')', "'", "'", '?']
original string: こんにちは世界
chunks: []


#### Now let's merge all the above patterns

In [42]:
pattern = r'\s+|\p{L}+|\p{N}+|[^\s\p{L}\p{N}]+'
apply_regex(pattern, samples)

original string: Hello.... world!!!!
chunks: ['Hello', '....', ' ', 'world', '!!!!']
original string: I'm learning BPE. 45320000
chunks: ['I', "'", 'm', ' ', 'learning', ' ', 'BPE', '.', ' ', '45320000']
original string: GPT-4 costs 200,000,000.0 euros.
chunks: ['GPT', '-', '4', ' ', 'costs', ' ', '200', ',', '000', ',', '000', '.', '0', ' ', 'euros', '.']
original string: Hallo, ich-lerne Deutsch.
chunks: ['Hallo', ',', ' ', 'ich', '-', 'lerne', ' ', 'Deutsch', '.']
original string: (Hello)   'world'
How are you?
chunks: ['(', 'Hello', ')', '   ', "'", 'world', "'", '\n', 'How', ' ', 'are', ' ', 'you', '?']
original string: こんにちは世界
chunks: ['こんにちは世界']


#### Right now i'm maps too i, ',' and 'm' separately but this is one word so let;s handle that

In [43]:
pattern = r"(?i:'s|'t|'re|'ve|'m|'ll|'d)" # case-insensitive match for common endings in contraptions
apply_regex(pattern, samples)

original string: Hello.... world!!!!
chunks: []
original string: I'm learning BPE. 45320000
chunks: ["'m"]
original string: GPT-4 costs 200,000,000.0 euros.
chunks: []
original string: Hallo, ich-lerne Deutsch.
chunks: []
original string: (Hello)   'world'
How are you?
chunks: []
original string: こんにちは世界
chunks: []


In [44]:
# now let's merge it with our other patterns
pattern = r"(?i:'s|'t|'re|'ve|'m|'ll|'d)|\s+|\p{L}+|\p{N}+|[^\s\p{L}\p{N}]+" # regex patterns are matched from left to right so contraptions get a chance to be merged before other words
apply_regex(pattern, samples)

original string: Hello.... world!!!!
chunks: ['Hello', '....', ' ', 'world', '!!!!']
original string: I'm learning BPE. 45320000
chunks: ['I', "'m", ' ', 'learning', ' ', 'BPE', '.', ' ', '45320000']
original string: GPT-4 costs 200,000,000.0 euros.
chunks: ['GPT', '-', '4', ' ', 'costs', ' ', '200', ',', '000', ',', '000', '.', '0', ' ', 'euros', '.']
original string: Hallo, ich-lerne Deutsch.
chunks: ['Hallo', ',', ' ', 'ich', '-', 'lerne', ' ', 'Deutsch', '.']
original string: (Hello)   'world'
How are you?
chunks: ['(', 'Hello', ')', '   ', "'", 'world', "'", '\n', 'How', ' ', 'are', ' ', 'you', '?']
original string: こんにちは世界
chunks: ['こんにちは世界']
